In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

sys.path.append('..')
sys.path.append(os.path.abspath(os.path.join('..', 'magnet-pinn')))
sys.path.append(os.path.abspath(os.path.join('..', 'neuraloperator')))

In [3]:
import torch

from torch.utils.data import DataLoader

from magnet_pinn.data.transforms import Compose, Crop, GridPhaseShift, SingleCoilZeroPhaseShift
from magnet_pinn.data.grid import MagnetGridIterator

VAL_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"

augmentation = Compose(
    [
        Crop(crop_size=(100, 100, 100)),
        SingleCoilZeroPhaseShift(num_coils=8)
    ]
)

val_set = MagnetGridIterator(VAL_DIR, transforms=augmentation, num_samples=8)
val_loader = iter(DataLoader(val_set, batch_size=1))

In [ ]:
from mrifield.models import UNet3D
from mrifield.train.lit_mrifield import LitMRIField
from magnet_pinn.utils import StandardNormalizer

CKPT = "baseline_unet/tkqjcw1e/baseline_unet_16M.ckpt"

model = UNet3D(in_channels=5, out_channels=12)

train_input_normalizer = StandardNormalizer.load_from_json("../results/normalization/train/input_normalization.json")
train_target_normalizer = StandardNormalizer.load_from_json("../results/normalization/train/target_normalization.json")
val_input_normalizer = StandardNormalizer.load_from_json("../results/normalization/val/input_normalization.json")
val_target_normalizer = StandardNormalizer.load_from_json("../results/normalization/val/target_normalization.json")

trained_model = LitMRIField.load_from_checkpoint(
    CKPT,
    model=model,
    train_input_normalizer=train_input_normalizer,
    train_target_normalizer=train_target_normalizer,
    val_input_normalizer=val_input_normalizer,
    val_target_normalizer=val_target_normalizer
)

trained_model.eval()

In [ ]:
import pytorch_lightning as pl

from torch.utils.data import DataLoader

from magnet_pinn.utils import StandardNormalizer
from magnet_pinn.data.transforms import Compose, Crop, GridPhaseShift
from magnet_pinn.data.grid import MagnetGridIterator
from magnet_pinn.data.utils import worker_init_fn

from neuralop.models import FNO
from mrifield.train.lit_mrifield import LitMRIField

TRAIN_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"
VAL_DIR = "../data/processed/batch_17/grid_voxel_size_4_data_type_float32"

model = FNO(n_modes=(16, 16, 16), in_channels=5, out_channels=12, hidden_channels=64, positional_embedding=None)

train_input_normalizer = StandardNormalizer.load_from_json(f"{TRAIN_DIR}/normalization/input_normalization.json")
train_target_normalizer = StandardNormalizer.load_from_json(f"{TRAIN_DIR}/normalization/target_normalization.json")
val_input_normalizer = StandardNormalizer.load_from_json(f"{VAL_DIR}/normalization/input_normalization.json")
val_target_normalizer = StandardNormalizer.load_from_json(f"{VAL_DIR}/normalization/target_normalization.json")

augmentation = Compose(
    [
        Crop(crop_size=(100, 100, 100)),
        GridPhaseShift(num_coils=8)
    ]
)

lit_model = LitMRIField(model, train_input_normalizer, train_target_normalizer, val_input_normalizer, val_target_normalizer)

train_set = MagnetGridIterator(TRAIN_DIR, transforms=augmentation, num_samples=100)
val_set = MagnetGridIterator(VAL_DIR, transforms=augmentation, num_samples=100)

train_loader = DataLoader(train_set, batch_size=4, num_workers=16, worker_init_fn=worker_init_fn)
val_loader = DataLoader(val_set, batch_size=4, num_workers=16, worker_init_fn=worker_init_fn)

trainer = pl.Trainer(accelerator="cpu", devices=1, log_every_n_steps=100, max_epochs=10)
trainer.fit(model=lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)